## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf
tf.enable_eager_execution()

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd

In [0]:
data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/cb/prices.csv')

### Check all columns in the dataset

In [5]:
data.columns

Index([u'date', u'symbol', u'open', u'close', u'low', u'high', u'volume'], dtype='object')

### Drop columns `date` and  `symbol`

In [0]:
data = data.drop(["date","symbol"], axis = 1)

In [7]:
data.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [8]:
data_new = data.iloc[:1000]
data_new["volume"] = data_new["volume"]/1000000
data_new.head(5)

/usr/local/lib/python2.7/dist-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split
x = data.drop(["volume"], axis = 1)
y = data[["volume"]]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)


#### Convert Training and Test Data to numpy float32 arrays


In [0]:
import numpy as np
x_train = x_train.to_numpy('float32')

In [0]:
y_train = y_train.to_numpy('float32')
x_test = x_test.to_numpy('float32')
y_test = y_test.to_numpy('float32')

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import normalize
normalized_x_train = normalize(np.vstack([x_train]),norm='l2',axis=1)
normalized_x_test = normalize(np.vstack([x_test]),norm='l2',axis=1)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.random_normal(shape=[4,1], name="Weights")
b = tf.random_normal(shape=[1],name="Bias")

In [0]:
#y = tf.add(tf.matmul(np.array(normalized_x_train),W),b,name='output')
#y_test_pred = tf.add(tf.matmul(np.array(normalized_x_test),W),b,name='output')

#loss_train = tf.reduce_mean(tf.square(y-y_train),name='Loss')
#loss_test = tf.reduce_mean(tf.square(y_test_pred-y_test),name='Loss')

2.Define a function to calculate prediction

In [0]:
def fn_prediction(x,W,b):
  y= tf.add(tf.matmul(np.array(x),W),b,name='output')
  return y

def fn_loss(y,y_pred):
  loss = tf.reduce_mean(tf.square(y-y_pred),name='Loss')
  return loss

3.Loss (Cost) Function [Mean square error]

In [0]:
def fn_train(x,y,W,b,learn_rate):
  with tf.GradientTape() as t:
    t.watch([W,b])
    y_pred = fn_prediction(x,W,b)
    loss = fn_loss(y,y_pred)
    dc_dw,dc_db = t.gradient(loss,[W,b])
    print(dc_dw)
    W= W -learn_rate * dc_dw
    b= b-learn_rate * dc_db
    return W,b


4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [47]:
for i in range(100):
  W,b = fn_train(normalized_x_train,y_train,W,b,0.01)
  fn_train(normalized_x_train,y_train,W,b,0.01)
  y_pred = fn_prediction(normalized_x_train,W,b)
  train_loss = fn_loss(y_train,y_pred)
  print("training loss at step :",i,"is",train_loss)
  if i % 5 == 0:
   print("training loss at 5th step :",i,"is",train_loss)

tf.Tensor(
[[ -1635.0146]
 [  1394.84  ]
 [ 15796.864 ]
 [-15010.154 ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -1633.8712]
 [  1395.9714]
 [ 15797.985 ]
 [-15009.046 ]], shape=(4, 1), dtype=float32)
('training loss at step :', 0, 'is', <tf.Tensor: id=24351, shape=(), dtype=float32, numpy=152654880000000.0>)
('training loss at 5th step :', 0, 'is', <tf.Tensor: id=24351, shape=(), dtype=float32, numpy=152654880000000.0>)
tf.Tensor(
[[ -1633.8712]
 [  1395.9714]
 [ 15797.985 ]
 [-15009.046 ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -1632.9678]
 [  1396.8888]
 [ 15798.8545]
 [-15008.165 ]], shape=(4, 1), dtype=float32)
('training loss at step :', 1, 'is', <tf.Tensor: id=24431, shape=(), dtype=float32, numpy=152654860000000.0>)
tf.Tensor(
[[ -1632.9678]
 [  1396.8888]
 [ 15798.8545]
 [-15008.165 ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -1632.0868]
 [  1397.7443]
 [ 15799.77  ]
 [-15007.276 ]], shape=(4, 1), dtype=float32)
('training loss at step :', 2, 'is', <tf.Tensor: id=24

### Get the shapes and values of W and b

In [49]:
for i in range(100):
  W,b = fn_train(normalized_x_train,y_train,W,b,0.01)
  fn_train(normalized_x_train,y_train,W,b,0.01)
  y_pred = fn_prediction(normalized_x_train,W,b)
  train_loss = fn_loss(y_train,y_pred)
  print("shape of W at  :",i,"is",W)
  print("shape of b at  :",i,"is",b)


tf.Tensor(
[[ -1609.537 ]
 [  1420.1013]
 [ 15821.294 ]
 [-14983.81  ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -1609.4017]
 [  1420.244 ]
 [ 15821.443 ]
 [-14983.662 ]], shape=(4, 1), dtype=float32)
('shape of W at  :', 0, 'is', <tf.Tensor: id=32225, shape=(4, 1), dtype=float32, numpy=
array([[1355251.8],
       [1343501.5],
       [1270878.6],
       [1423118.9]], dtype=float32)>)
('shape of b at  :', 0, 'is', <tf.Tensor: id=32228, shape=(1,), dtype=float32, numpy=array([2698479.2], dtype=float32)>)
tf.Tensor(
[[ -1609.4017]
 [  1420.244 ]
 [ 15821.443 ]
 [-14983.662 ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -1609.5002]
 [  1420.1322]
 [ 15821.343 ]
 [-14983.775 ]], shape=(4, 1), dtype=float32)
('shape of W at  :', 1, 'is', <tf.Tensor: id=32305, shape=(4, 1), dtype=float32, numpy=
array([[1355267.9],
       [1343487.2],
       [1270720.4],
       [1423268.8]], dtype=float32)>)
('shape of b at  :', 1, 'is', <tf.Tensor: id=32308, shape=(1,), dtype=float32, numpy=array([2698480

### Model Prediction on 1st Examples in Test Dataset

In [51]:
for i in range(100):
  W,b = fn_train(normalized_x_test,y_test,W,b,0.01)
  fn_train(normalized_x_test,y_test,W,b,0.01)
  y_pred = fn_prediction(normalized_x_test,W,b)
  train_loss = fn_loss(y_test,y_pred)
  if i % 1 == 0:
    print("training loss at 1st step :",i,"is",train_loss)
    print("y_pred loss at 1st step :",i,"is",y_pred) 

tf.Tensor(
[[ -2111.7856]
 [  1101.7484]
 [ 15427.51  ]
 [-16701.113 ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -2081.9202]
 [  1131.6223]
 [ 15457.039 ]
 [-16670.78  ]], shape=(4, 1), dtype=float32)
('training loss at 1st step :', 0, 'is', <tf.Tensor: id=48111, shape=(), dtype=float32, numpy=163486170000000.0>)
('y_pred loss at 1st step :', 0, 'is', <tf.Tensor: id=48106, shape=(255380, 1), dtype=float32, numpy=
array([[5439303. ],
       [5438604. ],
       [5439109.5],
       ...,
       [5438598. ],
       [5438950.5],
       [5438826.5]], dtype=float32)>)
tf.Tensor(
[[ -2081.9202]
 [  1131.6223]
 [ 15457.039 ]
 [-16670.78  ]], shape=(4, 1), dtype=float32)
tf.Tensor(
[[ -2053.4587]
 [  1160.1758]
 [ 15485.294 ]
 [-16642.14  ]], shape=(4, 1), dtype=float32)
('training loss at 1st step :', 1, 'is', <tf.Tensor: id=48191, shape=(), dtype=float32, numpy=163486150000000.0>)
('y_pred loss at 1st step :', 1, 'is', <tf.Tensor: id=48186, shape=(255380, 1), dtype=float32, numpy=
array([[54

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [0]:
dataset = pd.read_csv('/content/drive/My Drive/Colab Notebooks/cb/11_Iris.csv')

In [0]:
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import np_utils
from sklearn.preprocessing import LabelEncoder

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
dataset = pd.get_dummies(dataset,columns=["Species"])


In [0]:
X = dataset.iloc[:,1:5]
Y = dataset.iloc[:,5:8]

### Splitting the data into feature set and target set

In [0]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:
	# create model
model = tf.keras.Sequential()
#model.add(tf.keras.layers.Dense(8, input_dim=4, activation='relu'))
model.add(tf.keras.layers.Dense(3, input_shape=(4,),activation='softmax'))
# Compile model
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

In [106]:
model.fit(x_train, y_train, validation_data=(x_test,y_test),epochs=150, batch_size=10)

W0721 13:28:57.286921 140246524905344 deprecation.py:323] From /usr/local/lib/python2.7/dist-packages/tensorflow/python/ops/math_grad.py:1250: where (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Train on 105 samples, validate on 45 samples
Epoch 1/150
105/105 [==============================] - 1s 5ms/sample - loss: 2.1824 - acc: 0.3429 - val_loss: 1.2510 - val_acc: 0.5111
Epoch 2/150
105/105 [==============================] - 0s 192us/sample - loss: 1.2276 - acc: 0.2476 - val_loss: 1.0705 - val_acc: 0.5111
Epoch 3/150
105/105 [==============================] - 0s 172us/sample - loss: 1.0987 - acc: 0.3524 - val_loss: 1.0269 - val_acc: 0.4000
Epoch 4/150
105/105 [==============================] - 0s 177us/sample - loss: 1.0208 - acc: 0.3810 - val_loss: 1.0730 - val_acc: 0.3778
Epoch 5/150
105/105 [==============================] - 0s 183us/sample - loss: 0.9796 - acc: 0.4857 - val_loss: 0.9244 - val_acc: 0.5333
Epoch 6/150
105/105 [==============================] - 0s 176us/sample - loss: 0.9167 - acc: 0.5810 - val_loss: 0.8713 - val_acc: 0.4444
Epoch 7/150
105/105 [==============================] - 0s 171us/sample - loss: 0.8767 - acc: 0.5429 - val_loss: 0.9114 - val_acc: 0.600

### Model Training 

In [118]:
y_predict=model.predict_classes(x_train)
scores = model.evaluate(x_train, y_train)
scores[1]*100

105/105 [==============================] - 0s 99us/sample - loss: 0.2954 - acc: 0.9619


96.1904764175415

### Model Prediction

In [119]:
scores = model.evaluate(x_test, y_test)
scores[1]*100

45/45 [==============================] - 0s 231us/sample - loss: 0.3421 - acc: 0.9111


91.11111164093018

### Save the Model

In [0]:
model.save('Keras')

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?